# Backorder / Stockout-Risk Prediction — Real Dataset

This notebook uses the uploaded `clu.zip` dataset (`train.csv`, `test.csv`, `sample_Submission.csv`) from the product backorder prediction task.

**Target:** `went_on_backorder` (0 = no backorder, 1 = backorder).

**Important for your livestock FYP:** this is real general supply-chain backorder data, not livestock-feed inventory data. Results are useful as a supply-chain proof of concept, but must not be described as validated cattle-feed stockout performance.

The `Validation` column already marks a held-out validation subset in `train.csv`; we use it for evaluation and do not feed it to the model. We compare a Dummy baseline, Logistic Regression and Random Forest, then train a final model on all labelled rows and generate test predictions.

In [ ]:
# 1. Imports
import os, glob, zipfile, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, average_precision_score, precision_recall_curve,
    precision_recall_fscore_support
)
from sklearn.inspection import permutation_importance
import joblib
from IPython.display import display, FileLink

RANDOM_STATE = 42
warnings.filterwarnings("ignore")
print("Imports ready")

## 2. Locate and load the dataset

In Kaggle, upload `clu.zip` as a private Dataset and attach it through **Add Input**. This cell searches the attached input folders for `train.csv`, `test.csv`, and `sample_Submission.csv`. If you extracted the CSVs directly, that works too.

In [ ]:
# Search Kaggle inputs and current directory for files
search_roots = ["/kaggle/input", "/kaggle/working", "."]
train_paths, test_paths, sample_paths, zip_paths = [], [], [], []
for root in search_roots:
    if os.path.exists(root):
        train_paths += glob.glob(os.path.join(root, "**", "train.csv"), recursive=True)
        test_paths += glob.glob(os.path.join(root, "**", "test.csv"), recursive=True)
        sample_paths += glob.glob(os.path.join(root, "**", "sample_Submission.csv"), recursive=True)
        zip_paths += glob.glob(os.path.join(root, "**", "clu.zip"), recursive=True)

# If only the ZIP is available, extract it
if not train_paths and zip_paths:
    extract_dir = "/kaggle/working/clu_extracted"
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(zip_paths[0]) as z:
        z.extractall(extract_dir)
    train_paths = glob.glob(os.path.join(extract_dir, "**", "train.csv"), recursive=True)
    test_paths = glob.glob(os.path.join(extract_dir, "**", "test.csv"), recursive=True)
    sample_paths = glob.glob(os.path.join(extract_dir, "**", "sample_Submission.csv"), recursive=True)

if not train_paths or not test_paths:
    raise FileNotFoundError(
        "Could not find train.csv and test.csv. In Kaggle, click Add Input and attach the Dataset containing clu.zip, "
        "or upload/extract the three CSV files into the notebook."
    )

train_path = train_paths[0]
test_path = test_paths[0]
sample_path = sample_paths[0] if sample_paths else None

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sample_df = pd.read_csv(sample_path) if sample_path else None

print("Train:", train_df.shape, "| Test:", test_df.shape)
print("Target distribution:")
display(train_df["went_on_backorder"].value_counts().rename(index={0:"No backorder", 1:"Backorder"}))
print("Columns:")
display(pd.DataFrame({"column": train_df.columns, "dtype": train_df.dtypes.astype(str).values}))
display(train_df.head())

## 3. Use the supplied validation split

The dataset includes `Validation` in `train.csv`. We use `Validation == 0` for fitting and `Validation == 1` for evaluation. We drop `Validation` and `sku` from model inputs: `Validation` is split metadata, while `sku` is an identifier rather than a generalizable operational signal.

In [ ]:
TARGET = "went_on_backorder"
SPLIT_COL = "Validation"
DROP_COLS = [TARGET, SPLIT_COL, "sku"]

if SPLIT_COL in train_df.columns and train_df[SPLIT_COL].nunique() >= 2:
    fit_df = train_df[train_df[SPLIT_COL] == 0].copy()
    valid_df = train_df[train_df[SPLIT_COL] == 1].copy()
    split_note = "Using provided Validation flag: 0=fit, 1=held-out validation"
else:
    from sklearn.model_selection import train_test_split
    fit_df, valid_df = train_test_split(
        train_df, test_size=0.25, random_state=RANDOM_STATE,
        stratify=train_df[TARGET]
    )
    split_note = "Validation flag unavailable; using stratified random split"

feature_cols = [c for c in train_df.columns if c not in DROP_COLS]
X_fit, y_fit = fit_df[feature_cols], fit_df[TARGET].astype(int)
X_valid, y_valid = valid_df[feature_cols], valid_df[TARGET].astype(int)
X_comp_test = test_df.reindex(columns=feature_cols)

print(split_note)
print("Fit rows:", len(fit_df), "| Validation rows:", len(valid_df))
print("Positive rate — fit:", round(y_fit.mean(), 4), "| validation:", round(y_valid.mean(), 4))
print("Features:", feature_cols)

## 4. Train and compare models

Backorders are a minority class, so accuracy alone is misleading. We report precision, recall, F1, ROC-AUC and average precision (PR-AUC proxy). Higher recall means catching more actual backorders; precision indicates how many flagged items really are backorders.

In [ ]:
numeric_features = feature_cols
preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), numeric_features)
])

models = {
    "Dummy baseline": Pipeline([
        ("prep", preprocess),
        ("model", DummyClassifier(strategy="most_frequent"))
    ]),
    "Logistic Regression": Pipeline([
        ("prep", preprocess),
        ("model", LogisticRegression(
            max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
        ))
    ]),
    "Random Forest": Pipeline([
        ("prep", preprocess),
        ("model", RandomForestClassifier(
            n_estimators=300, min_samples_leaf=3, class_weight="balanced_subsample",
            random_state=RANDOM_STATE, n_jobs=-1
        ))
    ])
}

rows = []
fitted_models = {}
for name, model in models.items():
    model.fit(X_fit, y_fit)
    fitted_models[name] = model
    pred = model.predict(X_valid)
    proba = model.predict_proba(X_valid)[:, 1] if hasattr(model, "predict_proba") else None
    p, r, f1, _ = precision_recall_fscore_support(
        y_valid, pred, average="binary", zero_division=0
    )
    rows.append({
        "model": name,
        "precision_backorder": p,
        "recall_backorder": r,
        "f1_backorder": f1,
        "roc_auc": roc_auc_score(y_valid, proba) if proba is not None else np.nan,
        "average_precision": average_precision_score(y_valid, proba) if proba is not None else np.nan
    })

comparison = pd.DataFrame(rows).sort_values("average_precision", ascending=False)
display(comparison)
best_name = comparison.iloc[0]["model"]
best_model = fitted_models[best_name]
print("Best by average precision:", best_name)

In [ ]:
# Detailed validation report for the selected model
valid_pred = best_model.predict(X_valid)
valid_proba = best_model.predict_proba(X_valid)[:, 1]
print(classification_report(
    y_valid, valid_pred, target_names=["No backorder", "Backorder"],
    digits=4, zero_division=0
))
print("ROC-AUC:", round(roc_auc_score(y_valid, valid_proba), 4))
print("Average precision:", round(average_precision_score(y_valid, valid_proba), 4))

ConfusionMatrixDisplay.from_predictions(
    y_valid, valid_pred, display_labels=["No backorder", "Backorder"]
)
plt.title(f"Validation confusion matrix — {best_name}")
plt.show()

precision, recall, thresholds = precision_recall_curve(y_valid, valid_proba)
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision–Recall curve — {best_name}")
plt.grid(True)
plt.show()

## 5. Inspect feature importance

Permutation importance is computed on the held-out validation set using average precision. This is an association/diagnostic tool, not proof that a feature causes backorders.

In [ ]:
perm = permutation_importance(
    best_model, X_valid, y_valid, n_repeats=5,
    random_state=RANDOM_STATE, scoring="average_precision", n_jobs=-1
)
importance = pd.DataFrame({
    "feature": feature_cols,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)
display(importance)
importance.sort_values("importance_mean").plot(
    x="feature", y="importance_mean", kind="barh", legend=False,
    title="Permutation importance on validation data"
)
plt.xlabel("Change in average precision when shuffled")
plt.tight_layout()
plt.show()

## 6. Generate a risk score for each test item

The competition `test.csv` has no target column. We generate a predicted backorder probability and a class label for each test row. This is a probability from this model and dataset, not a livestock-specific risk score.

In [ ]:
test_probability = best_model.predict_proba(X_comp_test)[:, 1]
test_prediction = (test_probability >= 0.5).astype(int)

test_results = test_df.copy()
test_results["predicted_backorder_probability"] = test_probability
test_results["predicted_went_on_backorder"] = test_prediction
display(test_results[["sku", "predicted_backorder_probability", "predicted_went_on_backorder"]].head(10))
test_results.to_csv("/kaggle/working/backorder_test_predictions.csv", index=False)

## 7. Prepare competition-format submission (if sample file is present)

The sample submission determines the exact required columns. We inspect it and construct a file using the target predictions. Check the printed columns before submitting, because competitions can use different submission formats.

In [ ]:
if sample_df is not None:
    print("Sample submission columns:", sample_df.columns.tolist())
    display(sample_df.head())
    submission = sample_df.copy()
    # Common case: sample has one target column and rows align with test.csv
    if TARGET in submission.columns and len(submission) == len(test_df):
        submission[TARGET] = test_probability
        submission.to_csv("/kaggle/working/backorder_submission.csv", index=False)
        print("Saved /kaggle/working/backorder_submission.csv with probability predictions.")
    else:
        print("Sample format does not match the common target-column case; inspect columns and adapt before submitting.")
else:
    print("No sample submission file found; predictions are saved in backorder_test_predictions.csv.")

## 8. Save model and outputs

In [ ]:
out_dir = "/kaggle/working/backorder_model_outputs"
os.makedirs(out_dir, exist_ok=True)

# Fit final selected model on all labelled rows, excluding metadata and identifier.
X_all = train_df[feature_cols]
y_all = train_df[TARGET].astype(int)
final_model = fitted_models[best_name]
final_model.fit(X_all, y_all)

joblib.dump(final_model, os.path.join(out_dir, "backorder_risk_model.joblib"))
comparison.to_csv(os.path.join(out_dir, "model_comparison.csv"), index=False)
importance.to_csv(os.path.join(out_dir, "feature_importance.csv"), index=False)
test_results.to_csv(os.path.join(out_dir, "test_predictions.csv"), index=False)
with open(os.path.join(out_dir, "model_metadata.json"), "w") as f:
    json.dump({
        "dataset": "Product backorder competition dataset (clu.zip)",
        "target": TARGET,
        "selected_model": best_name,
        "validation_split": split_note,
        "fit_rows_for_validation": int(len(fit_df)),
        "validation_rows": int(len(valid_df)),
        "feature_columns": feature_cols,
        "warning": "General supply-chain backorder model; not validated for livestock-feed stockouts."
    }, f, indent=2)

zip_path = "/kaggle/working/backorder_prediction_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in os.listdir(out_dir):
        z.write(os.path.join(out_dir, fn), arcname=fn)
print("Results saved:", zip_path)
display(FileLink(zip_path))

## 9. Interpretation and limitations

- This dataset contains product-level backorder examples from a general supply chain.
- It is **not a livestock or cattle-feed dataset**.
- A backorder is related to supply availability, but it is not identical to a farm's physical stockout.
- Report validation metrics from the held-out split, not training metrics.
- For livestock-specific claims, collect daily feed inventory, consumption, purchase orders, actual delivery dates and confirmed stockout outcomes, then retrain and evaluate on later dates.